<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 🚀 Apache Beam Bulk ML Inference with Gemini, ShieldGemma 2 & Google Trends on Dataflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashwin-sampathkumar/ApacheBeam-Dataflow-Demos/blob/main/notebooks/beam_bulk_ml_inference_gemini_shieldgemma.ipynb)

### Overview
This production-grade codelab demonstrates **Apache Beam's Bulk ML Inference (`RunInference`)** combining **built-in remote Gemini model handlers** with **local GPU safety moderation using ShieldGemma 2**:
1. **Source**: Pulls trending search queries from **Google Trends** using **BigQuery** (`bigquery-public-data.google_trends.top_terms`).
2. **Pre-Shuffle Combiner Lifting**: Uses `beam.combiners.Sample.FixedSizePerKey(10)` instead of raw `GroupByKey()` to aggregate top search terms per city (`dma_name`) on mapper workers prior to shuffle.
3. **RunInference Stage 1 (Built-in `GeminiModelHandler` — Structured Text Generation)**: Uses `GeminiModelHandler` (`generate_from_string`) with Vertex AI Structured Outputs (`response_schema`) and optional **Vertex Flex API** (`use_vertex_flex_api`) to synthesize a cultural summary and image prompt.
4. **RunInference Stage 2 (Built-in `GeminiModelHandler` — Image Generation)**: Uses `GeminiModelHandler` (`generate_image_from_strings_and_images` with `gemini-2.5-flash-image`) chained via `with_preprocess_fn` / `with_postprocess_fn`—preserving city context **without a `CoGroupByKey()` shuffle**.
5. **RunInference Stage 3 (Local Vectorized GPU Inference — `ShieldGemma 2` + Resource Hints + DLQ)**: Runs true batched local inference with **`google/shieldgemma-2-4b-it`** (`ShieldGemma2ForImageClassification`), Dataflow Right-Fitting resource hints (`min_ram="32GB"`, `nvidia-l4` GPU), and native `RunInference.with_exception_handling()` Dead-Letter Queue (`RunInferenceDLQ`).
6. **Fail-Closed Conditional GCS Routing**: Routes verified safe images to the primary **Approved GCS Bucket**, policy-violating or unverified images to a **separate Quarantine GCS Bucket**, and inference exceptions to a **DLQ path**.

---

### Pipeline Architecture (Zero-Shuffle Linear Inference Chain)
```text
┌────────────────────────────────────────────────────────┐
│ BigQuery Source: Google Trends Public Dataset          │  SELECT * FROM `bigquery-public-data.google_trends.top_terms`
└───────────────────────────┬────────────────────────────┘
                            │
                            ▼
┌────────────────────────────────────────────────────────┐
│ beam.combiners.Sample.FixedSizePerKey(10)              │  Pre-shuffle combiner lifting (top 10 terms per dma_name)
└───────────────────────────┬────────────────────────────┘
                            │
                            ▼
┌────────────────────────────────────────────────────────┐
│ RunInference Stage 1: Built-in GeminiModelHandler      │  generate_from_string + JSON response_schema + Flex API
│ + with_preprocess_fn / with_postprocess_fn             │  Emits: {city, terms, summary, image_prompt}
└───────────────────────────┬────────────────────────────┘
                            │  (No CoGroupByKey shuffle!)
                            ▼
┌────────────────────────────────────────────────────────┐
│ RunInference Stage 2: Built-in GeminiModelHandler      │  generate_image_from_strings_and_images (gemini-2.5-flash-image)
│ + with_preprocess_fn / with_postprocess_fn             │  Emits: {city, summary, image_prompt, image_bytes}
└───────────────────────────┬────────────────────────────┘
                            │  (No CoGroupByKey shuffle!)
                            ▼
┌────────────────────────────────────────────────────────┐
│ RunInference Stage 3: Local ShieldGemma 2 (Vectorized) │  ShieldGemma2ForImageClassification (B*P batched forward pass)
│ + Right-Fitting GPU Resource Hints + .with_exception_  │  Evaluates: dangerous, sexual, violence policies
│   handling() (RunInferenceDLQ)                         │
└─────────────┬─────────────────────────────┬────────────┘
              │                             │ Exceptions (DLQ)
              ▼                             ▼
    ┌───────────────────┐         ┌──────────────────────┐
    │ Passed Safety?    │         │ DLQ GCS Path         │
    ├─────────┬─────────┤         │ gs://<unsafe>/dlq/   │
    │ YES     │ NO      │         └──────────────────────┘
    ▼         ▼         │
┌──────────┐ ┌──────────┴───────────────┐
│ Safe GCS │ │ Quarantine GCS Bucket    │
│ Bucket   │ │ gs://<unsafe-bucket>/    │
└──────────┘ └──────────────────────────┘
```

## 💡 Production Dataflow & Apache Beam `RunInference` Best Practices

This codelab showcases key patterns used by production Dataflow ML pipelines:

1. **Built-in `GeminiModelHandler` with Structured Outputs & Flex API**: Uses `apache_beam.ml.inference.gemini_inference.GeminiModelHandler` with `response_schema` (via `inference_args`) to guarantee valid JSON responses without brittle regex parsing, and supports `use_vertex_flex_api=True` for discounted batch throughput.
2. **Zero-Shuffle Context Chaining (`with_preprocess_fn` & `with_postprocess_fn`)**: Instead of stripping context before inference and re-joining large image bytes via `CoGroupByKey()`, each `KeyedModelHandler` uses `with_preprocess_fn` and `with_postprocess_fn` to carry the city metadata dictionary across all 3 stages without extra network shuffles.
3. **Pre-Shuffle Combiner Lifting**: Uses `beam.combiners.Sample.FixedSizePerKey(10)` instead of `GroupByKey()` so mapper workers bound per-city term lists before shuffling across the network.
4. **True Vectorized GPU Batching in `ShieldGemma2ModelHandler`**: Passes the entire batch of $B$ images across $P$ safety policies (`B * P` image-policy pairs) into a single `ShieldGemma2Processor` + `ShieldGemma2ForImageClassification` forward pass on the GPU, reshaping `outputs.probabilities` to `(B, P, 2)`.
5. **Dataflow Right-Fitting (`get_resource_hints`)**: Declares GPU and memory requirements (`min_ram="32GB"`, `accelerator="type:nvidia-l4;count:1;install-nvidia-driver"`) directly on `ShieldGemma2ModelHandler.get_resource_hints()` so Dataflow only provisions GPU accelerators for the local model stage.
6. **Native Dead-Letter Queue (`RunInference.with_exception_handling()`) & Fail-Closed Moderation**: Captures inference/preprocessing/postprocessing exceptions into a `RunInferenceDLQ` PCollection and supports fail-closed safety routing so unverified images never leak into the approved bucket.
7. **Worker Dependency Packaging (`requirements.txt`)**: Automatically generates a minimal worker `requirements.txt` for `--requirements_file` so remote Dataflow workers install `google-genai`, `transformers>=4.50.0`, and `accelerate`.

## Step 1: Install Dependencies
Google Colab already pre-installs `torch`, `transformers`, `accelerate`, `pillow`, `google-cloud-bigquery`, and `google-cloud-storage`.

We only install **`apache-beam[gcp]`** and **`google-genai`** (using `uv pip` when available or `pip --upgrade-strategy only-if-needed`), avoiding slow dependency resolver backtracking across Colab's pre-installed libraries.

> [!IMPORTANT]
> **Prerequisites & Model Access:**
> 1. **ShieldGemma 2** (`google/shieldgemma-2-4b-it`) is a gated model on Hugging Face. You must accept the Gemma terms at [Hugging Face: google/shieldgemma-2-4b-it](https://huggingface.co/google/shieldgemma-2-4b-it) and generate a Read Token under your Hugging Face Settings.
> 2. Recommended: Add your token to Google Colab Secrets (the key icon 🔑 in the left sidebar) as `HF_TOKEN`.

In [ ]:
import importlib
import importlib.util
import shutil
import subprocess
import sys
from typing import Any


def install_packages(pkgs):
    if not pkgs:
        return
    print(f"📦 Installing {pkgs}...")
    if shutil.which("uv"):
        subprocess.check_call(["uv", "pip", "install", "--system", "--quiet", *pkgs])
    else:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--quiet",
            "--upgrade-strategy", "only-if-needed", *pkgs,
        ])


# Detect the in-memory Pillow C-extension (_imaging) version pre-loaded by the Colab kernel
_pil_core_version = None
try:
    from PIL import _imaging
    _pil_core_version = getattr(_imaging, "PILLOW_VERSION", None)
except Exception:
    pass

# 1. Only install packages not already pre-installed in Colab (apache-beam[gcp], google-genai, bitsandbytes).
#    Pin pillow to _pil_core_version so pip/uv never upgrades Pillow out of sync with _imaging in RAM.
missing_pkgs = []
if importlib.util.find_spec("apache_beam") is None:
    missing_pkgs.append("apache-beam[gcp]")
if importlib.util.find_spec("google.genai") is None:
    missing_pkgs.append("google-genai")
if importlib.util.find_spec("bitsandbytes") is None:
    missing_pkgs.append("bitsandbytes>=0.43.0")
if missing_pkgs and _pil_core_version:
    missing_pkgs.append(f"pillow=={_pil_core_version}")

install_packages(missing_pkgs)

# 2. Ensure Pillow imports cleanly WITHOUT popping `PIL` or `PIL.Image` from `sys.modules`
#    (popping `PIL.Image` creates a duplicate `PIL.Image.Image` class in memory that breaks
#    `isinstance(img, PIL.Image.Image)` inside `transformers` and `torchvision`).
if "PIL._typing" in sys.modules and not hasattr(sys.modules["PIL._typing"], "_Ink"):
    setattr(sys.modules["PIL._typing"], "_Ink", Any)

try:
    from PIL import Image, ImageDraw  # noqa: F401
except ImportError:
    _target_pillow = f"pillow=={_pil_core_version}" if _pil_core_version else "pillow"
    print(f"🔧 Synchronizing {_target_pillow} on disk with loaded C extension...")
    install_packages(["--force-reinstall", "--no-deps", _target_pillow])
    if "PIL" in sys.modules:
        importlib.reload(sys.modules["PIL"])
    if "PIL._typing" in sys.modules and not hasattr(sys.modules["PIL._typing"], "_Ink"):
        setattr(sys.modules["PIL._typing"], "_Ink", Any)
    from PIL import Image, ImageDraw  # noqa: F401

# 3. Verify pre-installed transformers includes ShieldGemma2ForImageClassification (requires >=4.50.0)
import transformers
if not hasattr(transformers, "ShieldGemma2ForImageClassification"):
    install_packages(["transformers>=4.50.0"])

# Synchronize PIL module reference in transformers/torchvision if reloaded earlier in this session
import PIL
import transformers.image_utils as _tf_img_utils
_tf_img_utils.PIL = PIL

# 4. Write a minimal requirements.txt for remote DataflowRunner workers
with open("requirements.txt", "w") as req_file:
    req_file.write(
        "google-genai>=1.16.1\n"
        "transformers>=4.50.0\n"
        "accelerate>=0.26.0\n"
        "bitsandbytes>=0.43.0\n"
        "pillow\n"
    )

print(f"✅ Dependencies verified and ready! (Pillow {Image.__version__})")

## Step 2: Authenticate & Configure Environment
Authenticate with Google Cloud and specify your Project ID, Region, Primary GCS Bucket (for safe outputs), Quarantine GCS Bucket (for outputs failing ShieldGemma 2 safety checks), and Hugging Face Token (for downloading `google/shieldgemma-2-4b-it`).

In [ ]:
import os
import subprocess
import tempfile
import time
import urllib.request
import google.auth
import google.auth.transport.requests

# -------------------------------------------------------------
# 1. Configuration Parameters
# -------------------------------------------------------------
# Replace with your Google Cloud Project ID and GCS bucket
PROJECT_ID = "YOUR_PROJECT_ID"  # @param {type:"string"}
REGION = "us-central1"           # @param {type:"string"}

# Primary GCS bucket for images & metadata that PASS ShieldGemma 2 safety checks
GCS_BUCKET = "gs://YOUR_BUCKET_NAME/bulk_inference_demo/safe_output"  # @param {type:"string"}

# Separate Quarantine GCS bucket for images & metadata that FAIL ShieldGemma 2 safety checks (and DLQ records)
GCS_UNSAFE_BUCKET = "gs://YOUR_BUCKET_NAME/bulk_inference_demo/quarantine_output"  # @param {type:"string"}

# Execution Mode: 'DirectRunner' (runs locally in Colab) or 'DataflowRunner' (runs on Google Cloud Dataflow)
RUNNER = "DirectRunner"  # @param ["DirectRunner", "DataflowRunner"]

# Built-in Apache Beam Gemini Models & Local ShieldGemma 2 Model
GEMINI_TEXT_MODEL = "gemini-2.5-flash"         # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash"]
GEMINI_IMAGE_MODEL = "gemini-2.5-flash-image"  # @param ["gemini-2.5-flash-image"]
USE_VERTEX_FLEX_API = False                    # @param {type:"boolean"}

SHIELDGEMMA_MODEL = "google/shieldgemma-2-4b-it"  # @param ["google/shieldgemma-2-4b-it"]
# Precision for ShieldGemma 2: "auto" uses 4-bit NF4 on <=16 GB GPUs (T4) and bfloat16 on >=20 GB GPUs (L4/A100)
SHIELDGEMMA_PRECISION = "auto"                 # @param ["auto", "4bit", "8bit", "bfloat16"]
SAFETY_THRESHOLD = 0.5                         # @param {type:"number"}

# When True (production recommended), any image that cannot be verified by ShieldGemma 2
# (e.g. missing HF_TOKEN or unavailable GPU) is marked is_safe=False and routed to GCS_UNSAFE_BUCKET.
FAIL_CLOSED_ON_UNVERIFIED = True               # @param {type:"boolean"}

# Hugging Face Read Token for gated google/shieldgemma-2-4b-it weights
# Priority: Colab Secrets (userdata) -> Environment Variable -> Direct Input
HF_TOKEN = ""  # @param {type:"string"}
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        HF_TOKEN = ""
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Validation warnings
if PROJECT_ID == "YOUR_PROJECT_ID":
    print("⚠️ Notice: Please set your GCP PROJECT_ID before running on Google Cloud.")
if "YOUR_BUCKET_NAME" in GCS_BUCKET or "YOUR_BUCKET_NAME" in GCS_UNSAFE_BUCKET:
    print("⚠️ Notice: Please configure your GCS_BUCKET and GCS_UNSAFE_BUCKET.")
if not HF_TOKEN:
    print("⚠️ Notice: No HF_TOKEN found. ShieldGemma 2 requires a Hugging Face token with access to google/shieldgemma-2-4b-it.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID


def _reset_beam_gcp_credentials() -> None:
    try:
        from apache_beam.internal.gcp.auth import _Credentials
        _Credentials._credentials_init = False
        _Credentials._credentials = None
    except Exception:
        pass


def _is_token_accepted_by_gcp(token: str) -> bool:
    """Tests whether the OAuth2 access token is accepted by Google Cloud APIs (HTTP != 401)."""
    if not token or PROJECT_ID == "YOUR_PROJECT_ID":
        return False
    req = urllib.request.Request(
        f"https://bigquery.googleapis.com/bigquery/v2/projects/{PROJECT_ID}/datasets?maxResults=1",
        headers={"Authorization": f"Bearer {token}"},
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            return resp.status == 200
    except urllib.error.HTTPError as http_err:
        return http_err.code != 401
    except Exception:
        return False


def _interactive_gcloud_adc_login(target_adc_path: str) -> None:
    """Runs `gcloud auth application-default login` using Python `input()` so Colab renders the prompt box."""
    fd, temp_log = tempfile.mkstemp()
    proc = subprocess.Popen(
        [
            "gcloud",
            "auth",
            "application-default",
            "login",
            "--no-launch-browser",
            "--quiet",
        ],
        stdin=subprocess.PIPE,
        stdout=fd,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
    )
    try:
        prompt_text = ""
        for _ in range(150):
            time.sleep(0.2)
            os.fsync(fd)
            with open(temp_log, "r") as f:
                prompt_text = f.read()
            if "https://" in prompt_text or proc.poll() is not None:
                break
        time.sleep(0.3)
        os.fsync(fd)
        with open(temp_log, "r") as f:
            prompt_text = f.read()

        print(prompt_text)
        if proc.poll() is None:
            auth_code = input("🔑 Paste the authorization code here and press Enter: ").strip()
            proc.communicate(auth_code + "\n")
    finally:
        os.close(fd)
        if os.path.exists(temp_log):
            os.remove(temp_log)


def _ensure_valid_gcp_credentials() -> None:
    if PROJECT_ID == "YOUR_PROJECT_ID":
        print("ℹ️ Skipping GCP credential check until PROJECT_ID is configured.")
        return

    # 1. Check existing saved user ADC file
    default_adc_path = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
    for candidate_path in [
        os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", ""),
        default_adc_path,
        "/content/.adc/adc.json",
    ]:
        if candidate_path and os.path.isfile(candidate_path):
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = candidate_path
            try:
                creds, _ = google.auth.default(
                    scopes=["https://www.googleapis.com/auth/cloud-platform"]
                )
                creds.refresh(google.auth.transport.requests.Request())
                if _is_token_accepted_by_gcp(creds.token):
                    _reset_beam_gcp_credentials()
                    print(f"✅ Active user ADC credentials verified ({candidate_path})!")
                    return
            except Exception:
                pass
            os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS", None)

    # 2. Check ambient google.auth.default() (e.g. VM or Cloud Shell service account)
    try:
        creds, _ = google.auth.default(
            scopes=["https://www.googleapis.com/auth/cloud-platform"]
        )
        creds.refresh(google.auth.transport.requests.Request())
        if _is_token_accepted_by_gcp(creds.token):
            _reset_beam_gcp_credentials()
            print(
                f"✅ Valid GCP Application Default Credentials active "
                f"(principal: {getattr(creds, 'service_account_email', 'user')})!"
            )
            return
    except Exception:
        pass

    # 3. Try standard Colab browser popup auth
    try:
        from google.colab import auth
        saved_vp = os.environ.pop("VERTEX_PRODUCT", None)
        try:
            auth.authenticate_user(clear_output=False, project_id=PROJECT_ID)
        finally:
            if saved_vp is not None:
                os.environ["VERTEX_PRODUCT"] = saved_vp
        creds, _ = google.auth.default(
            scopes=["https://www.googleapis.com/auth/cloud-platform"]
        )
        creds.refresh(google.auth.transport.requests.Request())
        if _is_token_accepted_by_gcp(creds.token):
            _reset_beam_gcp_credentials()
            print("✅ Authenticated via Colab browser popup!")
            return
    except Exception:
        pass

    # 4. Fallback: interactive gcloud auth application-default login
    print("⚠️ GCP runtime token is unauthenticated for BigQuery.")
    print("👉 Click the OAuth link printed below, sign in, and paste the verification code into the input box:")
    _interactive_gcloud_adc_login(default_adc_path)

    if os.path.isfile(default_adc_path):
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = default_adc_path
        _reset_beam_gcp_credentials()
        print(f"✅ User ADC credentials saved and activated ({default_adc_path})!")


_ensure_valid_gcp_credentials()

print(f"Project:                   {PROJECT_ID}")
print(f"Region:                    {REGION}")
print(f"Safe GCS Output:           {GCS_BUCKET}")
print(f"Unsafe/Quarantine Output:  {GCS_UNSAFE_BUCKET}")
print(f"Gemini Text Model:         {GEMINI_TEXT_MODEL} (Flex API={USE_VERTEX_FLEX_API})")
print(f"Gemini Image Model:        {GEMINI_IMAGE_MODEL}")
print(f"ShieldGemma 2 Model:       {SHIELDGEMMA_MODEL} (precision={SHIELDGEMMA_PRECISION}, threshold={SAFETY_THRESHOLD}, fail_closed={FAIL_CLOSED_ON_UNVERIFIED})")
print(f"Runner:                    {RUNNER}")

## Step 3: BigQuery Google Trends Source
We use Google Trends public data in BigQuery (`bigquery-public-data.google_trends.top_terms`).
Here is the exact SQL query used as the source for our Apache Beam pipeline.


In [ ]:
from google.cloud import bigquery

# The BigQuery SQL query to pull city-level trends
BIGQUERY_SQL = """WITH RankedTerms AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY term
      ORDER BY RAND()
    ) AS term_row_num
  FROM `bigquery-public-data.google_trends.top_terms`
  WHERE week = "2026-09-20"
    -- AND dma_name LIKE ''
)
SELECT * EXCEPT (term_row_num)
FROM RankedTerms
WHERE term_row_num = 1
ORDER BY RAND()
LIMIT 10;"""

print("🔍 Previewing query results from BigQuery:")
bq_client = bigquery.Client(project=PROJECT_ID)
try:
    df_preview = bq_client.query(BIGQUERY_SQL).to_dataframe()
    display(df_preview.head(20))
    print(f"✅ Successfully retrieved {len(df_preview)} rows from BigQuery!")
    print(f"Columns available: {list(df_preview.columns)}")
except Exception as e:
    print(f"Note: Could not run direct preview ({e}). Ensure BigQuery API is enabled on your project.")

## Step 4: Configure Zero-Shuffle `GeminiModelHandler` & Vectorized `ShieldGemma 2` ModelHandler

We configure three `RunInference` stages chained via `.with_preprocess_fn(...)` and `.with_postprocess_fn(...)` (eliminating `CoGroupByKey()` shuffles):
1. **Stage 1 — Built-in `GeminiModelHandler` (`generate_from_string`) + Structured JSON Schema**:
   - Uses `apache_beam.ml.inference.gemini_inference.GeminiModelHandler` with `use_vertex_flex_api` and `GEMINI_TEXT_INFERENCE_ARGS` (`response_mime_type="application/json"`, `response_schema`).
   - `with_preprocess_fn` formats the city trends prompt while keeping `{"city": city, "terms": terms}` as the key; `with_postprocess_fn` attaches `summary` and `image_prompt` to the context dict.
2. **Stage 2 — Built-in `GeminiModelHandler` (`generate_image_from_strings_and_images`)**:
   - Uses `GeminiModelHandler` with `gemini-2.5-flash-image` to generate city artwork bytes and attaches `image_bytes` to the context dict via `with_postprocess_fn`.
3. **Stage 3 — Vectorized Local `ShieldGemma2ModelHandler` (`google/shieldgemma-2-4b-it`)  + Fail-Closed DLQ**:
   - Evaluates an entire batch of $B$ images across $P$ policies (`dangerous`, `sexual`, `violence`) in a **single batched GPU forward pass** (`shape: (B, P, 2)`).
   - Declares Dataflow Right-Fitting resource hints (`min_ram="32GB"`, `nvidia-l4` GPU) via `get_resource_hints()`.
   - Enforces **fail-closed moderation** (`FAIL_CLOSED_ON_UNVERIFIED`) so unverified or policy-violating records are quarantined to `GCS_UNSAFE_BUCKET`.

In [ ]:
import gc
import io
import json
import logging
import os
import re
import sys
import uuid
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np

# Help PyTorch avoid CUDA memory fragmentation on 15 GB Colab T4 GPUs
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Ensure Pillow imports cleanly WITHOUT popping PIL.Image from sys.modules
if "PIL._typing" in sys.modules and not hasattr(sys.modules["PIL._typing"], "_Ink"):
    setattr(sys.modules["PIL._typing"], "_Ink", Any)

from PIL import Image, ImageDraw
import PIL
import apache_beam as beam
from apache_beam.io.filesystems import FileSystems
from apache_beam.ml.inference.base import (
    KeyedModelHandler,
    ModelHandler,
    PredictionResult,
    RunInference,
)
from apache_beam.utils import multi_process_shared, shared as beam_shared
# Built-in Apache Beam ModelHandlers & request functions for Gemini
from apache_beam.ml.inference.gemini_inference import (
    GeminiModelHandler,
    generate_from_string,
)
try:
    from apache_beam.ml.inference.gemini_inference import (
        generate_image_from_strings_and_images,
    )
except ImportError:
    def generate_image_from_strings_and_images(
        model_name: str, batch: Sequence[Any], model: Any, inference_args: Dict[str, Any]
    ):
        return model.models.generate_content(
            model=model_name, contents=batch, **inference_args
        )

import torch
from transformers import AutoProcessor, ShieldGemma2ForImageClassification
try:
    from transformers import BitsAndBytesConfig
except ImportError:
    BitsAndBytesConfig = None
import transformers.image_utils as _tf_image_utils
import transformers.image_transforms as _tf_image_transforms

# Synchronize PIL references across transformers & torchvision in case PIL was reloaded earlier in this session
_tf_image_utils.PIL = PIL
_tf_image_transforms.PIL = PIL


# -------------------------------------------------------------
# 1. Stage 1 Preprocess/Postprocess & Structured Output Config
# -------------------------------------------------------------
GEMINI_TEXT_INFERENCE_ARGS: Dict[str, Any] = {
    "config": {
        "response_mime_type": "application/json",
        "response_schema": {
            "type": "OBJECT",
            "properties": {
                "summary": {"type": "STRING"},
                "image_prompt": {"type": "STRING"},
            },
            "required": ["summary", "image_prompt"],
        },
        "temperature": 0.7,
    }
}


def preprocess_for_gemini_text(
    city_terms_pair: Tuple[str, Iterable[str]]
) -> Tuple[Dict[str, Any], str]:
    """Formats city search terms into a prompt and carries context in the key."""
    city, terms = city_terms_pair
    terms_list = list(terms)[:10]
    terms_text = "\n".join([f"- {t}" for t in terms_list])

    prompt = f"""You are a visionary cultural anthropologist and art director.
Here are the top trending search topics in "{city}" for the week:
{terms_text}

Synthesize these trends and respond with valid JSON containing exactly two keys:
1. "summary": A vibrant, punchy 2-sentence cultural summary capturing the city's current pulse, mood, and obsessions.
2. "image_prompt": A vivid, photorealistic cinematic art prompt describing a captivating visual scene that embodies the spirit and trends of {city}. Do not include any text, letters, or logos in the image prompt."""
    context = {"city": city, "terms": terms_list}
    return (context, prompt)


def postprocess_gemini_text(
    element: Tuple[Dict[str, Any], PredictionResult]
) -> Dict[str, Any]:
    """Extracts structured JSON summary & image_prompt and enriches the city context."""
    context, pred = element
    city = context["city"]
    raw_text = ""
    try:
        response = pred.inference
        if isinstance(response, str):
            raw_text = response.strip()
        elif hasattr(response, "text") and response.text:
            raw_text = response.text.strip()
        elif hasattr(response, "parts") and response.parts:
            raw_text = "".join(
                p.text for p in response.parts if getattr(p, "text", None)
            ).strip()
        elif hasattr(response, "candidates") and response.candidates:
            parts = response.candidates[0].content.parts
            raw_text = "".join(
                p.text for p in parts if getattr(p, "text", None)
            ).strip()

        if raw_text.startswith("```json"):
            raw_text = raw_text[7:]
        if raw_text.startswith("```"):
            raw_text = raw_text[3:]
        if raw_text.endswith("```"):
            raw_text = raw_text[:-3]
        raw_text = raw_text.strip()

        parsed = json.loads(raw_text)
        summary = parsed.get("summary", "Trending updates.")
        image_prompt = parsed.get("image_prompt", f"Scenic cityscape of {city}")
    except Exception as e:
        logging.warning(f"Gemini output parse fallback for {city} ({e})")
        summary = (
            raw_text[:200]
            if raw_text
            else f"{city} is buzzing with cultural events, live entertainment, and trending local activities."
        )
        image_prompt = f"Atmospheric cinematic cityscape of {city} during golden hour"

    return {**context, "summary": summary, "image_prompt": image_prompt}


# -------------------------------------------------------------
# 2. Stage 2 Preprocess/Postprocess (Built-in Gemini Image Gen)
# -------------------------------------------------------------
def _render_fallback_card(prompt: str) -> bytes:
    """Renders a clean fallback PNG card if inline image data is unavailable."""
    img = Image.new("RGB", (768, 768), color=(26, 32, 44))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0, 0), (768, 14)], fill=(66, 133, 244))
    draw.rectangle([(0, 14), (768, 22)], fill=(234, 67, 53))
    draw.text((40, 50), "GOOGLE TRENDS x GEMINI BULK INFERENCE", fill=(160, 174, 192))
    draw.text((40, 90), "City Trend Art Card", fill=(247, 250, 252))

    words = prompt.split()
    lines, cur = [], []
    for w in words:
        cur.append(w)
        if len(" ".join(cur)) > 42:
            lines.append(" ".join(cur[:-1]))
            cur = [w]
    if cur:
        lines.append(" ".join(cur))

    y = 160
    draw.text((40, y), "Artistic Scene Prompt:", fill=(251, 188, 4))
    y += 35
    for line in lines[:10]:
        draw.text((40, y), line, fill=(226, 232, 240))
        y += 26

    draw.text((40, 710), "Powered by Apache Beam RunInference on Dataflow", fill=(128, 134, 139))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def preprocess_for_gemini_image(
    context: Dict[str, Any]
) -> Tuple[Dict[str, Any], str]:
    """Extracts the synthesized image_prompt while preserving full city context in the key."""
    return (context, context["image_prompt"])


def postprocess_gemini_image(
    element: Tuple[Dict[str, Any], PredictionResult]
) -> Dict[str, Any]:
    """Extracts generated PNG image bytes from built-in GeminiModelHandler response."""
    context, pred = element
    city = context["city"]
    prompt_str = context["image_prompt"]
    image_bytes = None
    image_source = "gemini-2.5-flash-image"
    try:
        response = pred.inference
        parts = getattr(response, "parts", None)
        if not parts and getattr(response, "candidates", None):
            parts = response.candidates[0].content.parts
        if parts:
            for part in parts:
                inline_data = getattr(part, "inline_data", None)
                if inline_data is not None and getattr(inline_data, "data", None):
                    img = Image.open(io.BytesIO(inline_data.data))
                    buf = io.BytesIO()
                    img.save(buf, format="PNG")
                    image_bytes = buf.getvalue()
                    break
    except Exception as e:
        logging.warning(f"Gemini image extraction notice for {city} ({e}); using fallback renderer.")

    if image_bytes is None:
        image_bytes = _render_fallback_card(prompt_str)
        image_source = "fallback-card-renderer"

    return {**context, "image_bytes": image_bytes, "image_source": image_source}


# -------------------------------------------------------------
# 3. Stage 3: Vectorized Local ShieldGemma 2 ModelHandler + large_model=True
# -------------------------------------------------------------
class _SharedShieldGemma2:
    """Callable wrapper around ShieldGemma 2 model and processor for `large_model=True`.

    Memory optimizations for 15 GB Colab T4 GPUs:
    1. Supports 4-bit NF4 (`BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)`)
       or `bfloat16` precision so model weights take ~2.8 GiB instead of ~8.5 GiB.
    2. Passes `logits_to_keep=1` and `use_cache=False`, and slices `hidden_states[:, -1:, :]`
       before `lm_head` so the 262k-vocab projection never allocates a 3.0 GiB full-sequence logit tensor.
    3. Evaluates 1 policy at a time (`batch_size=1`) on <=16 GB GPUs and clears CUDA cache
       (`gc.collect()` + `torch.cuda.empty_cache()`) after every inference batch.
    """

    DEFAULT_POLICIES = ["dangerous", "sexual", "violence"]

    def __init__(self, model: Any, processor: Any, precision_mode: str = "bfloat16"):
        self.model = model
        self.processor = processor
        self.precision_mode = precision_mode
        self._patch_lm_head_last_token_only()

    def _patch_lm_head_last_token_only(self) -> None:
        """Wraps `lm_head` so 3D hidden_states `(B, seq_len, hidden)` are sliced to `(B, 1, hidden)`
        BEFORE the 262,144-vocab linear projection, preventing a 3.0 GiB logit tensor allocation
        even on transformers versions that ignore `logits_to_keep=1` inside `Gemma3ForConditionalGeneration`.
        """
        try:
            inner = getattr(self.model, "model", None)
            lm_head = getattr(inner, "lm_head", None)
            if lm_head is not None and not getattr(lm_head, "_last_token_patched", False):
                orig_forward = lm_head.forward

                def _sliced_lm_head_forward(hidden_states: torch.Tensor) -> torch.Tensor:
                    if hidden_states.ndim == 3 and hidden_states.shape[1] > 1:
                        hidden_states = hidden_states[:, -1:, :]
                    return orig_forward(hidden_states)

                lm_head.forward = _sliced_lm_head_forward
                lm_head._last_token_patched = True
        except Exception:
            pass

    def _score_single_image_policies(
        self,
        rgb_image: np.ndarray,
        policies: Sequence[str],
        custom_policies: Optional[Mapping[str, str]],
        safety_threshold: float,
        policy_chunk_size: int,
    ) -> Dict[str, Any]:
        policy_scores: Dict[str, float] = {}
        policies_list = list(policies)

        for p_start in range(0, len(policies_list), policy_chunk_size):
            sub_policies = policies_list[p_start : p_start + policy_chunk_size]
            proc_kwargs: Dict[str, Any] = {
                "images": [rgb_image],
                "policies": sub_policies,
                "return_tensors": "pt",
            }
            if custom_policies:
                proc_kwargs["custom_policies"] = dict(custom_policies)

            inputs = self.processor(**proc_kwargs).to(self.model.device)
            with torch.inference_mode():
                outputs = self.model(**inputs, logits_to_keep=1, use_cache=False)
                probs = outputs.probabilities.detach().cpu().float().view(len(sub_policies), 2).tolist()

            del inputs, outputs
            for p_idx, policy_name in enumerate(sub_policies):
                policy_scores[policy_name] = round(float(probs[p_idx][0]), 4)

        violated_policies = [
            policy for policy, score in policy_scores.items() if score >= safety_threshold
        ]
        return {
            "is_safe": len(violated_policies) == 0,
            "violated_policies": violated_policies,
            "policy_scores": policy_scores,
            "threshold": safety_threshold,
            "evaluation_mode": f"shieldgemma-2-{self.precision_mode}",
        }

    def __call__(
        self,
        batch: Sequence[bytes],
        policies: Sequence[str],
        custom_policies: Optional[Mapping[str, str]],
        safety_threshold: float,
    ) -> List[Dict[str, Any]]:
        rgb_images = [
            np.array(Image.open(io.BytesIO(img_bytes)).convert("RGB"), copy=True)
            for img_bytes in batch
        ]

        # On <=16 GB GPUs (Colab T4), score 1 policy per forward pass (`policy_chunk_size=1`)
        # to keep peak vision + attention activation memory minimal; on >=20 GB GPUs, score all policies at once.
        policy_chunk_size = len(policies)
        if torch.cuda.is_available():
            total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            if total_vram_gb < 20.0:
                policy_chunk_size = 1

        all_verdicts: List[Dict[str, Any]] = []
        try:
            for rgb_img in rgb_images:
                verdict = self._score_single_image_policies(
                    rgb_img,
                    policies,
                    custom_policies,
                    safety_threshold,
                    policy_chunk_size=policy_chunk_size,
                )
                all_verdicts.append(verdict)
        finally:
            # Reviewer recommendation: explicitly run garbage collection and clear PyTorch CUDA cache
            # after inference so back-to-back batches in Colab never accumulate cached CUDA blocks.
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return all_verdicts


class ShieldGemma2ModelHandler(ModelHandler[bytes, PredictionResult, Any]):
    """Vectorized Apache Beam ModelHandler running local inference with ShieldGemma 2
    (google/shieldgemma-2-4b-it) with `large_model=True`, 4-bit/bfloat16 precision, and
    automatic per-batch CUDA cache clearing.
    """

    DEFAULT_POLICIES = ["dangerous", "sexual", "violence"]

    def __init__(
        self,
        model_id: str = "google/shieldgemma-2-4b-it",
        hf_token: Optional[str] = None,
        precision: str = "auto",
        safety_threshold: float = 0.5,
        fail_closed: bool = True,
        policies: Optional[Sequence[str]] = None,
        custom_policies: Optional[Mapping[str, str]] = None,
        min_batch_size: int = 1,
        max_batch_size: int = 1,
        large_model: bool = True,
    ):
        self.model_id = model_id
        self.hf_token = hf_token or None
        self.precision = precision
        self.safety_threshold = safety_threshold
        self.fail_closed = fail_closed
        self.policies = list(policies) if policies else list(self.DEFAULT_POLICIES)
        self.custom_policies = dict(custom_policies) if custom_policies else None
        self._large_model = large_model
        self._share_across_processes = large_model
        self._batching_kwargs = {
            "min_batch_size": min_batch_size,
            "max_batch_size": max_batch_size,
        }

    def share_model_across_processes(self) -> bool:
        """Enables `MultiProcessShared` model sharing across processes when `large_model=True`."""
        return self._large_model

    def batch_elements_kwargs(self) -> Mapping[str, Any]:
        return self._batching_kwargs

    def get_resource_hints(self) -> dict:
        """Declares Dataflow Right-Fitting hints so GPU workers are scoped to ShieldGemma 2."""
        return {
            "min_ram": "32GB",
            "accelerator": "type:nvidia-l4;count:1;install-nvidia-driver",
        }

    def load_model(self) -> Optional[_SharedShieldGemma2]:
        """Loads ShieldGemma2ForImageClassification in 4-bit NF4 (on T4) or bfloat16 (on L4/A100)."""
        try:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            device = "cuda" if torch.cuda.is_available() else "cpu"
            dtype = torch.bfloat16 if device == "cuda" else torch.float32

            # Determine whether to use 4-bit NF4 quantization (reduces weights from ~8.5 GiB to ~2.8 GiB)
            use_4bit = False
            use_8bit = False
            if device == "cuda" and BitsAndBytesConfig is not None:
                total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
                if self.precision == "4bit" or (self.precision == "auto" and total_vram_gb < 20.0):
                    use_4bit = True
                elif self.precision == "8bit":
                    use_8bit = True

            processor = AutoProcessor.from_pretrained(
                self.model_id,
                token=self.hf_token,
            )

            load_kwargs: Dict[str, Any] = {
                "torch_dtype": dtype,
                "device_map": {"": 0} if device == "cuda" else None,
                "low_cpu_mem_usage": True,
                "token": self.hf_token,
            }
            precision_label = "bfloat16" if device == "cuda" else "float32"

            if use_4bit:
                load_kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                )
                precision_label = "4bit-nf4-bfloat16"
            elif use_8bit:
                load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
                precision_label = "8bit-bfloat16"

            logging.info(
                f"Loading local ShieldGemma 2 model '{self.model_id}' on {device} ({precision_label})..."
            )
            try:
                model = ShieldGemma2ForImageClassification.from_pretrained(
                    self.model_id,
                    **load_kwargs,
                )
            except Exception as quant_err:
                if "quantization_config" in load_kwargs:
                    logging.warning(f"Quantized load fallback to bfloat16 ({quant_err})...")
                    load_kwargs.pop("quantization_config", None)
                    precision_label = "bfloat16"
                    model = ShieldGemma2ForImageClassification.from_pretrained(
                        self.model_id,
                        **load_kwargs,
                    )
                else:
                    raise

            if device == "cpu":
                model = model.to(torch.device("cpu"))
            model.eval()
            logging.info(f"✅ ShieldGemma 2 loaded successfully on {model.device} ({precision_label})!")
            return _SharedShieldGemma2(
                model=model,
                processor=processor,
                precision_mode=precision_label,
            )
        except Exception as e:
            logging.warning(
                f"ShieldGemma 2 local model load notice ({e}). "
                "Ensure you have accepted the Gemma license on Hugging Face and provided HF_TOKEN."
            )
            return None

    def run_inference(
        self,
        batch: Sequence[bytes],
        model: Any,
        inference_args: Optional[Dict[str, Any]] = None,
    ) -> Iterable[PredictionResult]:
        if not batch:
            return []

        if model is not None:
            try:
                verdicts = model(
                    batch,
                    self.policies,
                    self.custom_policies,
                    self.safety_threshold,
                )
                return [
                    PredictionResult(img_bytes, verdict, self.model_id)
                    for img_bytes, verdict in zip(batch, verdicts)
                ]
            except Exception as e:
                logging.error(f"ShieldGemma 2 batched inference error: {e}")
                if self.fail_closed:
                    raise
            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        # Unverified path when model weights/HF_TOKEN are unavailable
        unverified_is_safe = not self.fail_closed
        unverified_violations = ["unverified_model_unavailable"] if self.fail_closed else []
        unverified_scores = {policy: None for policy in self.policies}
        return [
            PredictionResult(
                img_bytes,
                {
                    "is_safe": unverified_is_safe,
                    "violated_policies": unverified_violations,
                    "policy_scores": unverified_scores,
                    "threshold": self.safety_threshold,
                    "evaluation_mode": "unverified-fail-closed" if self.fail_closed else "unverified-demo-fallback",
                },
                self.model_id,
            )
            for img_bytes in batch
        ]


# Clear any previous unquantized or stale ShieldGemma 2 singleton in MultiProcessShared so the
# memory-optimized model + patched lm_head takes effect immediately in this Colab session.
beam_shared._shared_map._keepalive = (None, None)
beam_shared._shared_map._cache_map.clear()
for _tag, _entry in list(multi_process_shared._process_level_singleton_manager.entries.items()):
    _obj = getattr(_entry, "obj", None)
    multi_process_shared._process_level_singleton_manager.entries.pop(_tag, None)
    del _obj, _entry
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    _free_gb, _total_gb = [b / (1024**3) for b in torch.cuda.mem_get_info()]
    print(f"🧹 GPU VRAM ready: {_free_gb:.2f} GiB free / {_total_gb:.2f} GiB total")


def preprocess_for_shieldgemma(
    context: Dict[str, Any]
) -> Tuple[Dict[str, Any], bytes]:
    """Extracts image_bytes for ShieldGemma 2 while preserving city context in the key."""
    return (context, context["image_bytes"])


def postprocess_shieldgemma(
    element: Tuple[Dict[str, Any], PredictionResult]
) -> Tuple[str, Dict[str, Any]]:
    """Attaches ShieldGemma 2 safety verdict to the final city record."""
    context, pred = element
    return (context["city"], {**context, "safety_check": pred.inference})


# -------------------------------------------------------------
# 4. Safety Router, GCS Sinks & DLQ Handler DoFns
# -------------------------------------------------------------
class RouteBySafetyDoFn(beam.DoFn):
    """Routes records to 'safe' (main output) or 'unsafe' (tagged output) based on ShieldGemma 2 verdict."""

    UNSAFE_TAG = "unsafe"

    def process(self, element: Tuple[str, Dict[str, Any]]):
        city, data = element
        safety_check = data.get("safety_check", {})
        is_safe = safety_check.get("is_safe", False)
        if is_safe:
            yield (city, data)
        else:
            yield beam.pvalue.TaggedOutput(self.UNSAFE_TAG, (city, data))


class SaveOutputsToGCSDoFn(beam.DoFn):
    """Writes images and JSON metadata (including ShieldGemma 2 safety scores) to a target GCS bucket."""

    def __init__(self, output_path: str, bucket_label: str = "safe"):
        self.output_path = output_path.rstrip("/")
        self.bucket_label = bucket_label

    def process(self, element: Tuple[str, Dict[str, Any]]):
        city, data = element
        clean_city_slug = re.sub(r"[^a-zA-Z0-9_-]", "_", city).strip("_")

        image_dest = f"{self.output_path}/images/{clean_city_slug}.png"
        meta_dest = f"{self.output_path}/metadata/{clean_city_slug}.json"

        image_bytes = data.get("image_bytes")
        if image_bytes:
            with FileSystems.create(image_dest) as f:
                f.write(image_bytes)

        payload = {
            "city": city,
            "status": self.bucket_label,
            "terms": data.get("terms", []),
            "summary": data.get("summary"),
            "image_prompt": data.get("image_prompt"),
            "image_source": data.get("image_source"),
            "image_gcs_uri": image_dest,
            "safety_check": data.get("safety_check", {}),
        }
        with FileSystems.create(meta_dest) as f:
            f.write(json.dumps(payload, indent=2).encode("utf-8"))

        yield (city, self.bucket_label, image_dest, meta_dest)


class SaveDLQErrorsToGCSDoFn(beam.DoFn):
    """Writes failed inference batches/elements (and any generated images) from RunInferenceDLQ to Quarantine GCS."""

    def __init__(self, dlq_output_path: str):
        self.dlq_output_path = dlq_output_path.rstrip("/")

    def process(self, failed_element: Tuple[Any, Any]):
        failed_input, error_msg = failed_element
        record_id = uuid.uuid4().hex[:10]
        dlq_dest = f"{self.dlq_output_path}/error_{record_id}.json"

        # If the failed batch already contains generated images from Stage 2, preserve them in DLQ
        saved_dlq_images = []
        try:
            items = failed_input if isinstance(failed_input, (list, tuple)) else [failed_input]
            for item in items:
                ctx = item[0] if isinstance(item, tuple) and len(item) == 2 and isinstance(item[0], dict) else (
                    item if isinstance(item, dict) else None
                )
                if isinstance(ctx, dict) and ctx.get("image_bytes"):
                    city_name = ctx.get("city", record_id)
                    slug = re.sub(r"[^a-zA-Z0-9_-]", "_", str(city_name)).strip("_")
                    img_uri = f"{self.dlq_output_path}/images/{slug}_{record_id}.png"
                    with FileSystems.create(img_uri) as img_f:
                        img_f.write(ctx["image_bytes"])
                    saved_dlq_images.append(img_uri)
        except Exception:
            pass

        payload = {
            "record_id": record_id,
            "error": str(error_msg),
            "saved_dlq_images": saved_dlq_images,
            "failed_input_repr": repr(failed_input)[:500],
        }
        with FileSystems.create(dlq_dest) as f:
            f.write(json.dumps(payload, indent=2).encode("utf-8"))
        yield dlq_dest

print("✅ Zero-shuffle ModelHandlers, ShieldGemma2ModelHandler (4-bit/bfloat16 + large_model=True + CUDA cache cleanup), Right-Fitting hints, and DLQ defined!")

## Step 5: Build and Run the Apache Beam Pipeline

The pipeline:
1. **Reads Google Trends from BigQuery** (`beam.io.ReadFromBigQuery`) and applies **pre-shuffle combiner lifting** (`beam.combiners.Sample.FixedSizePerKey(10)`).
2. **Chains Stage 1 (`GeminiModelHandler` Text)** $\rightarrow$ **Stage 2 (`GeminiModelHandler` Image)** $\rightarrow$ **Stage 3 (`ShieldGemma 2` Safety Check)** via `.with_preprocess_fn(...)` and `.with_postprocess_fn(...)`, eliminating any `CoGroupByKey()` shuffle of raw image bytes across worker nodes!
3. **Applies Right-Fitting Resource Hints** (`min_ram="32GB"`, `accelerator="type:nvidia-l4;count:1;install-nvidia-driver"`) scoped to the ShieldGemma 2 safety transform.
4. **Isolates Failures with Native DLQ** (`RunInference.with_exception_handling()`).
5. **Routes Output Fail-Closed**: Verified safe images to primary GCS, policy-violating and unverified images to Quarantine GCS, and runtime exceptions to the Quarantine DLQ.

In [ ]:
import datetime
import inspect
from apache_beam.options.pipeline_options import PipelineOptions, SetupOptions

# Reset Apache Beam's process-level cached GCP credentials so fresh ADC tokens are always used
try:
    from apache_beam.internal.gcp.auth import _Credentials
    _Credentials._credentials_init = False
    _Credentials._credentials = None
except Exception:
    pass

# 1. Configure Pipeline Options & Separate GCS Output Paths
run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"{GCS_BUCKET}/trends_inference_{run_ts}"
unsafe_output_dir = f"{GCS_UNSAFE_BUCKET}/trends_inference_{run_ts}"
dlq_output_dir = f"{unsafe_output_dir}/dlq"
job_name = f"beam-gemini-shieldgemma-trends-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

pipeline_args = [
    f"--runner={RUNNER}",
    f"--project={PROJECT_ID}",
    f"--region={REGION}",
    f"--temp_location={GCS_BUCKET}/temp",
    f"--staging_location={GCS_BUCKET}/staging",
    f"--job_name={job_name}",
]

if RUNNER == "DataflowRunner":
    pipeline_args.extend([
        "--save_main_session=True",
        "--requirements_file=requirements.txt",
        "--max_num_workers=5",
        "--machine_type=g2-standard-8",
        "--dataflow_service_options=worker_accelerator=type:nvidia-l4;count:1;install-nvidia-driver",
        "--experiments=use_runner_v2,enable_recommendation_service",
    ])

pipeline_options = PipelineOptions(pipeline_args)
pipeline_options.view_as(SetupOptions).save_main_session = True

print(f"🚀 Launching Pipeline using {RUNNER}...")
print(f"✅ Safe destination output path:   {output_dir}")
print(f"🚨 Unsafe quarantine output path:  {unsafe_output_dir}")
print(f"🛑 Dead-Letter Queue (DLQ) path:   {dlq_output_dir}")

# 2. Configure Chained ModelHandlers (with_preprocess_fn & with_postprocess_fn)
# Only pass `use_vertex_flex_api` if supported by the installed apache-beam version
_gemini_extra_kwargs = (
    {"use_vertex_flex_api": USE_VERTEX_FLEX_API}
    if "use_vertex_flex_api" in inspect.signature(GeminiModelHandler.__init__).parameters
    else {}
)

gemini_text_handler = (
    KeyedModelHandler(
        GeminiModelHandler(
            model_name=GEMINI_TEXT_MODEL,
            request_fn=generate_from_string,
            project=PROJECT_ID,
            location=REGION,
            max_batch_size=1,
            **_gemini_extra_kwargs,
        )
    )
    .with_preprocess_fn(preprocess_for_gemini_text)
    .with_postprocess_fn(postprocess_gemini_text)
)

gemini_image_handler = (
    KeyedModelHandler(
        GeminiModelHandler(
            model_name=GEMINI_IMAGE_MODEL,
            request_fn=generate_image_from_strings_and_images,
            project=PROJECT_ID,
            location=REGION,
            max_batch_size=1,
            **_gemini_extra_kwargs,
        )
    )
    .with_preprocess_fn(preprocess_for_gemini_image)
    .with_postprocess_fn(postprocess_gemini_image)
)

shieldgemma_handler = (
    KeyedModelHandler(
        ShieldGemma2ModelHandler(
            model_id=SHIELDGEMMA_MODEL,
            hf_token=HF_TOKEN,
            precision=SHIELDGEMMA_PRECISION,
            safety_threshold=SAFETY_THRESHOLD,
            fail_closed=FAIL_CLOSED_ON_UNVERIFIED,
            max_batch_size=1,
            large_model=True,
        )
    )
    .with_preprocess_fn(preprocess_for_shieldgemma)
    .with_postprocess_fn(postprocess_shieldgemma)
)

# 3. Build and Execute Zero-Shuffle Linear DAG
with beam.Pipeline(options=pipeline_options) as p:
    # Step 1: Read Google Trends from BigQuery & lift top-10 sample combiner pre-shuffle
    sampled_city_terms = (
        p
        | "ReadFromBigQuery" >> beam.io.ReadFromBigQuery(
            query=BIGQUERY_SQL,
            use_standard_sql=True,
        )
        | "FilterValidDMA" >> beam.Filter(
            lambda row: row.get("dma_name") is not None and str(row.get("dma_name", "")).strip() != ""
        )
        | "ExtractCityTerm" >> beam.Map(
            lambda row: (
                str(row["dma_name"]).strip(),
                f"{row.get('term', '')} (Rank: {row.get('rank', 'N/A')}, Score: {row.get('score', 'N/A')})",
            )
        )
        | "SampleTopTermsPerCity" >> beam.combiners.Sample.FixedSizePerKey(10)
    )

    # Step 2: RunInference Stage 1 - Built-in GeminiModelHandler (Structured JSON Text Generation)
    gemini_text_records = (
        sampled_city_terms
        | "RunGeminiTextInference" >> RunInference(
            gemini_text_handler,
            inference_args=GEMINI_TEXT_INFERENCE_ARGS,
        )
    )

    # Step 3: RunInference Stage 2 - Built-in GeminiModelHandler (Image Generation)
    gemini_image_records = (
        gemini_text_records
        | "RunGeminiImageInference" >> RunInference(gemini_image_handler)
    )

    # Step 4: RunInference Stage 3 - Local Vectorized ShieldGemma 2 Safety Check (large_model=True) + Native DLQ
    moderated_records, safety_dlq = (
        gemini_image_records
        | "RunShieldGemma2SafetyCheck" >> RunInference(
            shieldgemma_handler
        ).with_exception_handling()
    )

    # Step 5: Route Safe vs. Unsafe Records to Separate GCS Buckets
    routed_records = (
        moderated_records
        | "RouteByShieldGemmaSafety" >> beam.ParDo(RouteBySafetyDoFn()).with_outputs(
            RouteBySafetyDoFn.UNSAFE_TAG, main="safe"
        )
    )

    # Step 6a: Save verified safe records to Primary GCS Bucket
    _ = routed_records.safe | "SaveSafeToGCS" >> beam.ParDo(
        SaveOutputsToGCSDoFn(output_dir, bucket_label="safe")
    )

    # Step 6b: Save policy-violating / unverified records to Separate Quarantine GCS Bucket
    _ = routed_records[RouteBySafetyDoFn.UNSAFE_TAG] | "SaveUnsafeToGCS" >> beam.ParDo(
        SaveOutputsToGCSDoFn(unsafe_output_dir, bucket_label="unsafe_quarantine")
    )

    # Step 6c: Sink any RunInferenceDLQ exceptions to Quarantine DLQ path
    _ = safety_dlq.failed_inferences | "SaveSafetyDLQToGCS" >> beam.ParDo(
        SaveDLQErrorsToGCSDoFn(dlq_output_dir)
    )

print("🎉 Pipeline completed / submitted successfully!")
if RUNNER == "DataflowRunner":
    print(f"👉 Monitor Dataflow Job at: https://console.cloud.google.com/dataflow/jobs/{REGION}/{job_name}?project={PROJECT_ID}")

## Step 6: Interactive Output & Safety Audit Visualizer
Display the generated images, Gemini cultural summaries, and **ShieldGemma 2** policy violation scores from both the **Primary (Safe) GCS Bucket** and the **Quarantine (Unsafe) GCS Bucket** right here in Colab!

In [ ]:
from google.cloud import storage
from IPython.display import display, Image as IPImage
import json

storage_client = storage.Client(project=PROJECT_ID)


def render_gcs_bucket_results(target_dir: str, section_title: str):
    print(f"\n{'#' * 65}")
    print(f"{section_title}")
    print(f"Path: {target_dir}")
    print(f"{'#' * 65}")

    try:
        bucket_name = target_dir.replace("gs://", "").split("/")[0]
        base_prefix = "/".join(target_dir.replace("gs://", "").split("/")[1:])
        meta_prefix = f"{base_prefix}/metadata/"

        bucket = storage_client.bucket(bucket_name)
        meta_blobs = list(bucket.list_blobs(prefix=meta_prefix))

        if not meta_blobs:
            print(f"ℹ️ No records found in {target_dir}.")
            dlq_blobs = [
                b for b in bucket.list_blobs(prefix=f"{base_prefix}/dlq/")
                if b.name.endswith(".json")
            ]
            if dlq_blobs:
                print(f"⚠️ Found {len(dlq_blobs)} Dead-Letter Queue (DLQ) error record(s) in {target_dir}/dlq/:")
                sample_err = json.loads(dlq_blobs[0].download_as_text())
                print(f"   Sample DLQ Error: {str(sample_err.get('error', ''))[:300]}")
            return

        print(f"Found {len(meta_blobs)} city record(s):\n")
        for blob in meta_blobs:
            meta = json.loads(blob.download_as_text())
            city = meta.get("city")
            summary = meta.get("summary")
            image_prompt = meta.get("image_prompt")
            image_source = meta.get("image_source", "gemini-2.5-flash-image")
            image_uri = meta.get("image_gcs_uri")
            safety = meta.get("safety_check", {})
            is_safe = safety.get("is_safe", False)
            eval_mode = safety.get("evaluation_mode", "unknown")
            scores = safety.get("policy_scores", {})
            violated = safety.get("violated_policies", [])

            badge = (
                f"✅ PASSED SAFETY CHECKS ({eval_mode})"
                if is_safe
                else f"🚨 QUARANTINED [{eval_mode}] (Reason: {', '.join(violated)})"
            )

            print("==================================================")
            print(f"🏙️  CITY: {city}")
            print(f"🛡️  SHIELDGEMMA 2 VERDICT: {badge}")
            print(f"📊  POLICY VIOLATION PROBABILITIES: {json.dumps(scores)}")
            print(f"📝  GEMINI SUMMARY:\n{summary}")
            print(f"🎨  GEMINI IMAGE PROMPT ({image_source}):\n{image_prompt}")
            print(f"📦  GCS URI: {image_uri}")
            print("==================================================")

            image_blob_name = image_uri.replace(f"gs://{bucket_name}/", "")
            img_blob = bucket.blob(image_blob_name)
            if img_blob.exists():
                img_data = img_blob.download_as_bytes()
                display(IPImage(data=img_data))
            print("\n")
    except Exception as e:
        print(f"Error loading results from {target_dir}: {e}")


# 1. Inspect Safe / Approved Outputs
render_gcs_bucket_results(output_dir, "✅ APPROVED OUTPUTS (Primary GCS Bucket)")

# 2. Inspect Quarantined / Policy-Violating / Unverified Outputs
render_gcs_bucket_results(unsafe_output_dir, "🚨 QUARANTINED OUTPUTS (Unsafe GCS Bucket)")

## Step 7: Submitting as a Dataflow Job via CLI (Alternative)
You can also trigger this pipeline directly from Cloud Shell or your local development environment using the modular pipeline script (`pipeline/main.py`) with `--requirements_file=pipeline/requirements.txt` and Dataflow Right-Fitting for local **ShieldGemma 2** GPU inference:

```bash
python -m pipeline.main \
    --runner=DataflowRunner \
    --project=YOUR_GCP_PROJECT_ID \
    --region=us-central1 \
    --temp_location=gs://YOUR_GCS_BUCKET/temp \
    --staging_location=gs://YOUR_GCS_BUCKET/staging \
    --output_path=gs://YOUR_GCS_BUCKET/bulk_inference_demo/safe_output \
    --unsafe_output_path=gs://YOUR_GCS_BUCKET/bulk_inference_demo/quarantine_output \
    --requirements_file=pipeline/requirements.txt \
    --machine_type=g2-standard-8 \
    --dataflow_service_options="worker_accelerator=type:nvidia-l4;count:1;install-nvidia-driver" \
    --experiments=use_runner_v2,enable_recommendation_service \
    --job_name=dataflow-trends-gemini-shieldgemma
```